In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
#client using langchain
from langchain_openai import ChatOpenAI

llm_client= ChatOpenAI(
        model="gpt-4.1-mini",
        temperature=0.7,
)
response = llm_client.invoke("Tell me about NPCI")

In [ ]:
import sqlite3
from datetime import datetime, timedelta

DB_PATH = "./db.sqlite"

def init_database(path: str = DB_PATH):
    conn = sqlite3.connect(path)
    cur = conn.cursor()

    cur.executescript(
        """
        DROP TABLE IF EXISTS users;
        DROP TABLE IF EXISTS orders;
        DROP TABLE IF EXISTS order_items;
        DROP TABLE IF EXISTS returns;
        """
    )

    cur.executescript(
        """
        CREATE TABLE users (
            email TEXT PRIMARY KEY,
            full_name TEXT NOT NULL
        );

        CREATE TABLE orders (
            order_id TEXT PRIMARY KEY,
            user_email TEXT NOT NULL,
            order_date TEXT NOT NULL,
            status TEXT NOT NULL,
            total_amount REAL NOT NULL,
            FOREIGN KEY(user_email) REFERENCES users(email)
        );

        CREATE TABLE order_items (
            order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
            order_id TEXT NOT NULL,
            product_id TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            unit_price REAL NOT NULL,
            FOREIGN KEY(order_id) REFERENCES orders(order_id)
        );

        CREATE TABLE returns (
            return_id TEXT PRIMARY KEY,
            order_id TEXT NOT NULL,
            user_email TEXT NOT NULL,
            return_date TEXT NOT NULL,
            reason TEXT NOT NULL,
            status TEXT NOT NULL,
            refund_amount REAL,
            FOREIGN KEY(order_id) REFERENCES orders(order_id),
            FOREIGN KEY(user_email) REFERENCES users(email)
        );
        """
    )

    users = [
        ("good_user@example.com", "Good User"),
        ("fraudster@example.com", "Frequent Returner"),
    ]
    cur.executemany("INSERT INTO users VALUES (?, ?)", users)

    now = datetime.now()

    def d(days: int) -> str:
        return (now - timedelta(days=days)).strftime("%Y-%m-%d")

    orders = [
      ("ORD_GOOD_1", "good_user@example.com", d(5), "DELIVERED", 1299.99),
      ("ORD_GOOD_1_dup2", "good_user@example.com", d(5), "DELIVERED", 1299.99),
      ("ORD_GOOD_1_dup3", "good_user@example.com", d(5), "DELIVERED", 1299.99),
      ("ORD_GOOD_2", "good_user@example.com", d(20), "DELIVERED", 249.99),
      ("ORD_GOOD_2_dup2", "good_user@example.com", d(20), "DELIVERED", 249.99),
      ("ORD_GOOD_2_dup3", "good_user@example.com", d(20), "DELIVERED", 249.99),
      ("ORD_GOOD_3", "good_user@example.com", d(12), "DELIVERED", 120.00),
      ("ORD_GOOD_3_dup2", "good_user@example.com", d(12), "DELIVERED", 120.00),
      ("ORD_GOOD_3_dup3", "good_user@example.com", d(12), "DELIVERED", 120.00),
      ("ORD_GOOD_4", "good_user@example.com", d(32), "DELIVERED", 349.00),
      ("ORD_GOOD_4_dup2", "good_user@example.com", d(32), "DELIVERED", 349.00),
      ("ORD_GOOD_4_dup3", "good_user@example.com", d(32), "DELIVERED", 349.00),
      ("ORD_BAD_1", "fraudster@example.com", d(1), "DELIVERED", 899.99),
      ("ORD_BAD_1_dup2", "fraudster@example.com", d(1), "DELIVERED", 899.99),
      ("ORD_BAD_1_dup3", "fraudster@example.com", d(1), "DELIVERED", 899.99),
      ("ORD_BAD_2", "fraudster@example.com", d(25), "DELIVERED", 1099.00),
      ("ORD_BAD_2_dup2", "fraudster@example.com", d(25), "DELIVERED", 1099.00),
      ("ORD_BAD_2_dup3", "fraudster@example.com", d(25), "DELIVERED", 1099.00),
      ("ORD_BAD_3", "fraudster@example.com", d(3), "DELIVERED", 1299.99),
      ("ORD_BAD_3_dup2", "fraudster@example.com", d(3), "DELIVERED", 1299.99),
      ("ORD_BAD_3_dup3", "fraudster@example.com", d(3), "DELIVERED", 1299.99),
      ("ORD_BAD_4", "fraudster@example.com", d(8), "DELIVERED", 450.00),
      ("ORD_BAD_4_dup2", "fraudster@example.com", d(8), "DELIVERED", 450.00),
      ("ORD_BAD_4_dup3", "fraudster@example.com", d(8), "DELIVERED", 450.00),
      ("ORD_BAD_5", "fraudster@example.com", d(2), "DELIVERED", 2200.00),
      ("ORD_BAD_5_dup2", "fraudster@example.com", d(2), "DELIVERED", 2200.00),
      ("ORD_BAD_5_dup3", "fraudster@example.com", d(2), "DELIVERED", 2200.00),
    ]
    cur.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders)

    items = [
      (None, "ORD_GOOD_1", "SKU_LAPTOP", 1, 1299.99),
      (None, "ORD_GOOD_1_dup2", "SKU_LAPTOP", 1, 1299.99),
      (None, "ORD_GOOD_1_dup3", "SKU_LAPTOP", 1, 1299.99),
      (None, "ORD_GOOD_2", "SKU_MOUSE", 1, 249.99),
      (None, "ORD_GOOD_2_dup2", "SKU_MOUSE", 1, 249.99),
      (None, "ORD_GOOD_2_dup3", "SKU_MOUSE", 1, 249.99),
      (None, "ORD_GOOD_3", "SKU_CABLE", 2, 60.00),
      (None, "ORD_GOOD_3_dup2", "SKU_CABLE", 2, 60.00),
      (None, "ORD_GOOD_3_dup3", "SKU_CABLE", 2, 60.00),
      (None, "ORD_GOOD_4", "SKU_BACKPACK", 1, 349.00),
      (None, "ORD_GOOD_4_dup2", "SKU_BACKPACK", 1, 349.00),
      (None, "ORD_GOOD_4_dup3", "SKU_BACKPACK", 1, 349.00),
      (None, "ORD_BAD_1", "SKU_MONITOR", 1, 899.99),
      (None, "ORD_BAD_1_dup2", "SKU_MONITOR", 1, 899.99),
      (None, "ORD_BAD_1_dup3", "SKU_MONITOR", 1, 899.99),
      (None, "ORD_BAD_2", "SKU_WEBCAM", 1, 1099.00),
      (None, "ORD_BAD_2_dup2", "SKU_WEBCAM", 1, 1099.00),
      (None, "ORD_BAD_2_dup3", "SKU_WEBCAM", 1, 1099.00),
      (None, "ORD_BAD_3", "SKU_DESK", 1, 1299.99),
      (None, "ORD_BAD_3_dup2", "SKU_DESK", 1, 1299.99),
      (None, "ORD_BAD_3_dup3", "SKU_DESK", 1, 1299.99),
      (None, "ORD_BAD_4", "SKU_HEADSET", 2, 225.00),
      (None, "ORD_BAD_4_dup2", "SKU_HEADSET", 2, 225.00),
      (None, "ORD_BAD_4_dup3", "SKU_HEADSET", 2, 225.00),
      (None, "ORD_BAD_5", "SKU_GAMING_PC", 1, 2200.00),
      (None, "ORD_BAD_5_dup2", "SKU_GAMING_PC", 1, 2200.00),
      (None, "ORD_BAD_5_dup3", "SKU_GAMING_PC", 1, 2200.00),
    ]
    cur.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)", items)

    returns_data = [
      ("RET_BAD_PREV_1", "ORD_BAD_1", "fraudster@example.com", d(3), "changed my mind", "APPROVED", 350.00),
      ("RET_BAD_PREV_1_dup2", "ORD_BAD_1_dup2", "fraudster@example.com", d(3), "changed my mind", "APPROVED", 350.00),
      ("RET_BAD_PREV_1_dup3", "ORD_BAD_1_dup3", "fraudster@example.com", d(3), "changed my mind", "APPROVED", 350.00),
      ("RET_BAD_PREV_2", "ORD_BAD_1", "fraudster@example.com", d(10), "ordered by mistake", "APPROVED", 520.00),
      ("RET_BAD_PREV_2_dup2", "ORD_BAD_1_dup2", "fraudster@example.com", d(10), "ordered by mistake", "APPROVED", 520.00),
      ("RET_BAD_PREV_2_dup3", "ORD_BAD_1_dup3", "fraudster@example.com", d(10), "ordered by mistake", "APPROVED", 520.00),
      ("RET_BAD_PREV_3", "ORD_BAD_2", "fraudster@example.com", d(15), "don't need", "APPROVED", 650.00),
      ("RET_BAD_PREV_3_dup2", "ORD_BAD_2_dup2", "fraudster@example.com", d(15), "don't need", "APPROVED", 650.00),
      ("RET_BAD_PREV_3_dup3", "ORD_BAD_2_dup3", "fraudster@example.com", d(15), "don't need", "APPROVED", 650.00),
      ("RET_BAD_PREV_4", "ORD_BAD_3", "fraudster@example.com", d(5), "no reason", "APPROVED", 1200.00),
      ("RET_BAD_PREV_4_dup2", "ORD_BAD_3_dup2", "fraudster@example.com", d(5), "no reason", "APPROVED", 1200.00),
      ("RET_BAD_PREV_4_dup3", "ORD_BAD_3_dup3", "fraudster@example.com", d(5), "no reason", "APPROVED", 1200.00),
      ("RET_BAD_PREV_5", "ORD_BAD_3", "fraudster@example.com", d(1), "ordered by mistake", "APPROVED", 1299.99),
      ("RET_BAD_PREV_5_dup2", "ORD_BAD_3_dup2", "fraudster@example.com", d(1), "ordered by mistake", "APPROVED", 1299.99),
      ("RET_BAD_PREV_5_dup3", "ORD_BAD_3_dup3", "fraudster@example.com", d(1), "ordered by mistake", "APPROVED", 1299.99),
      ("RET_BAD_PREV_6", "ORD_BAD_5", "fraudster@example.com", d(1), "changed my mind", "APPROVED", 1800.00),
      ("RET_BAD_PREV_6_dup2", "ORD_BAD_5_dup2", "fraudster@example.com", d(1), "changed my mind", "APPROVED", 1800.00),
      ("RET_BAD_PREV_6_dup3", "ORD_BAD_5_dup3", "fraudster@example.com", d(1), "changed my mind", "APPROVED", 1800.00),
      ("RET_GOOD_PREV_1", "ORD_GOOD_1", "good_user@example.com", d(40), "arrived damaged", "APPROVED", 249.99),
      ("RET_GOOD_PREV_1_dup2", "ORD_GOOD_1_dup2", "good_user@example.com", d(40), "arrived damaged", "APPROVED", 249.99),
      ("RET_GOOD_PREV_1_dup3", "ORD_GOOD_1_dup3", "good_user@example.com", d(40), "arrived damaged", "APPROVED", 249.99),
      ("RET_GOOD_PREV_2", "ORD_GOOD_3", "good_user@example.com", d(6), "size too small", "APPROVED", 120.00),
      ("RET_GOOD_PREV_2_dup2", "ORD_GOOD_3_dup2", "good_user@example.com", d(6), "size too small", "APPROVED", 120.00),
      ("RET_GOOD_PREV_2_dup3", "ORD_GOOD_3_dup3", "good_user@example.com", d(6), "size too small", "APPROVED", 120.00),
    ]
    cur.executemany("INSERT INTO returns VALUES (?, ?, ?, ?, ?, ?, ?)", returns_data)

    conn.commit()
    conn.close()

init_database()
print("Fraud database ready at", DB_PATH)

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")
toolkit = SQLDatabaseToolkit(db=db, llm=model)
tools = toolkit.get_tools()

print("Available database tools:")
for tool in tools:
    print(f"- {tool.name}")
print()

In [ ]:
#fetch from DB using tools
shared_sql_instructions = """
SQL RULES (apply to every query):
- Use only the provided SQL tools.
- ALWAYS list tables, then inspect schema for tables you touch.
- Add LIMIT when listing rows unless the question says otherwise.
- Use the SQL query checker tool before executing a query.
- NEVER run INSERT, UPDATE, DELETE, DROP, or other DML.
"""

SHOPPER_EMAIL = "fraudster@example.com"
#ORDER_ID = "ORD_BAD_2"
#RETURN_REASON = "changed my mind"
#RETURN_ITEMS = [{"product_id": "SKU_WEBCAM", "quantity": 1}]


frequency_role = """
ROLE:
Return-frequency risk analyst on an e-commerce fraud team.

GOAL:
When given a shopper email, count their non-rejected returns in the last 30 days
and assign a frequency risk_score (0–100) using the rubric below.

BACKSTORY:
You investigate whether a customer is returning items too often — a common sign
of policy abuse. You query a read-only SQLite database with tables: users, orders,
order_items, returns. You focus ONLY on return counts and timing. Ignore refund
dollar amounts and return-reason text; other specialists on the team handle those.

WORKFLOW:
1. Query the returns table — count rows where:
   - user_email matches the shopper
   - return_date is within the last 30 days
   - status is NOT 'REJECTED'
2. Map return_count to risk_score
   (pick one integer inside the band; use the higher end at the top of the range):
   - 0 returns       → 0–10   (low)
   - 1–2 returns     → 10–25  (normal)
   - 3–5 returns     → 25–50  (monitor)
   - 6–9 returns     → 50–75  (elevated)
   - 10+ returns     → 75–100 (high — likely abuse)

OUTPUT FORMAT:
1. Write 2–4 sentences in plain language summarizing what you found.
2. End with ONE line of strict JSON only (no markdown fences). Example shape:
   {"check_type": "frequency", "risk_score": 0, "reason": "short explanation",
    "details": {"return_count": 0, "period_days": 30}}

JSON field rules:
- risk_score: integer 0–100 from the rubric (use the real count, not a placeholder)
- reason: one sentence citing the count and band
  (e.g. "12 returns in 30 days — high frequency")
- details.return_count: exact integer from your SQL query
- details.period_days: always 30
"""


frequency_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=shared_sql_instructions + frequency_role,
)

freq_result = frequency_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            f"Return ticket — frequency risk check\n"
            f"Shopper email: {SHOPPER_EMAIL}\n\n"
            "How many non-rejected returns has this shopper made in the last 30 days? "
            "Give a brief explanation and a frequency risk score."
        ),
    }]
})
print(freq_result["messages"][-1].content)